In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv('dsat.csv', dtype={352: 'str', 353: 'str'})
print(df.head())
print(df.shape)

In [ ]:
print(list(df['DESIDR1_SPECTYPE'].unique()))

df = df[
    (df['DESIDR1_SPECTYPE'] != "b'STAR  '") & 
    (df['ls_gaia_star_sn4'] == False) &
    (df['DESIDR1_SPECTYPE'].notna())
]
print(df.head())
df = df.drop(columns=['Unnamed: 0'])

print(df.head())
print(df.dtypes)
print(df.shape)

In [ ]:
df = df.select_dtypes(include=['float64'])
print(df.head())
print(df.dtypes)
print(df.shape)

In [ ]:
# eROSITA (рентгеновский)
# WISE (инфракрасный)
# Legacy Imaging Surveys (оптический)
# SDSS (оптический)
# PanSTARRS1 (оптический)
# GAIA (оптический)

with open('columns.txt', 'w') as f:
    for name in df.columns:
        f.write(name + '\n')
    
df.head(10).to_csv('first_10_rows.csv', index=False)

In [ ]:
with open('columns.txt', 'r') as f:
    for name in f.readlines():
        name = name.strip()
        missings = df[name].isnull().sum()
        if missings != 0:
            print(f"{name}: {missings}")

In [ ]:
def build_scatter_for_redshift(name):
    plt.figure(figsize=(10, 10))
    plt.scatter(df[name], df['DESIDR1_Z'], c=df['DESIDR1_Z'], s=10)
    plt.xlabel(name, fontsize=12)
    plt.ylabel('RedShift', fontsize=12)
    plt.show()

# with open('columns.txt', 'r') as f:
#     for name in f.readlines():
#         name = name.strip()
#         build_scatter_for_redshift(name)

In [ ]:
features = []
with open("used.txt", "r") as f:
    for line in f.readlines():
        line = line.strip()
        features.append(line)
print(features)

In [ ]:
for name in features:
    build_scatter_for_redshift(name)

In [ ]:
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna()

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
cols = df.select_dtypes(include=['float']).columns
df[cols] = scaler.fit_transform(df[cols])

In [ ]:
for name in features:
    build_scatter_for_redshift(name)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

X = df[features]
y = df['DESIDR1_Z']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=pd.qcut(y, q=5, labels=False)
)

print(y_train)

In [ ]:
!pip3 install catboost

In [ ]:
from catboost import CatBoostRegressor

cb_model = CatBoostRegressor(
    iterations=3000,
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=3,
    bagging_temperature=0.5,
    loss_function='RMSE',
    random_state=42,
    early_stopping_rounds=200,
    verbose=200
)

cb_model.fit(
    X_train, y_train,
    eval_set=(X_test, y_test),
    plot=True
)

y_pred_cb = cb_model.predict(X_test)

rmse_cb = np.sqrt(mean_squared_error(y_test, y_pred_cb))
mae_cb = mean_absolute_error(y_test, y_pred_cb)
r2_cb = r2_score(y_test, y_pred_cb)

print("\nCatBoost Results:")
print(f"RMSE: {rmse_cb:.4f}")
print(f"MAE:  {mae_cb:.4f}")
print(f"R²:   {r2_cb:.4f}")


In [ ]:

importances = cb_model.get_feature_importance()
fi = pd.DataFrame({
    "feature": X_train.columns,
    "importance": importances
}).sort_values("importance", ascending=False)

fi.to_csv("fi.txt", index=False, sep="\t")

In [ ]:
THRESHOLDS = [0.1, 0.2, 0.3, 0.4, 0.5]

fi = pd.read_csv("fi.txt", sep="\t")

for thr in THRESHOLDS:
    keep_features = fi[fi["importance"] > thr]["feature"].tolist()
    
    X_train_distill = X_train[keep_features]
    X_test_distill = X_test[keep_features]
    
    cb_model_distill = CatBoostRegressor(
        iterations=2000,
        depth=6,
        learning_rate=0.03,
        loss_function='RMSE',
        random_state=42,
        early_stopping_rounds=150,
        verbose=200
    )
    
    cb_model_distill.fit(
        X_train_distill, y_train,
        eval_set=(X_test_distill, y_test)
    )
    
    y_pred_cb_distill = cb_model_distill.predict(X_test_distill)
    
    rmse_cb_distill = np.sqrt(mean_squared_error(y_test, y_pred_cb_distill))
    mae_cb_distill = mean_absolute_error(y_test, y_pred_cb_distill)
    r2_cb_distill = r2_score(y_test, y_pred_cb_distill)

    print()
    print(f"CatBoost Results For THR={thr}:")
    print(f"RMSE: {rmse_cb_distill:.4f}")
    print(f"MAE:  {mae_cb_distill:.4f}")
    print(f"R²:   {r2_cb_distill:.4f}")
    print()

In [ ]:
from sklearn.model_selection import KFold

kfold = KFold(n_splits=5, shuffle=True, random_state=42)

rmse_scores, mae_scores, r2_scores = [], [], []

for fold, (train_idx, val_idx) in enumerate(kfold.split(X), 1):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = CatBoostRegressor(
        iterations=2000,
        depth=6,
        learning_rate=0.03,
        loss_function='RMSE',
        random_state=42,
        early_stopping_rounds=150,
        verbose=False
    )

    model.fit(X_tr, y_tr, eval_set=(X_val, y_val), verbose=False)
    y_pred = model.predict(X_val)

    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    mae = mean_absolute_error(y_val, y_pred)
    r2 = r2_score(y_val, y_pred)

    rmse_scores.append(rmse)
    mae_scores.append(mae)
    r2_scores.append(r2)

    print(f"Fold {fold}: RMSE={rmse:.4f}, MAE={mae:.4f}, R²={r2:.4f}")

print("\nMean over folds:")
print(f"RMSE: {np.mean(rmse_scores):.4f} ± {np.std(rmse_scores):.4f}")
print(f"MAE:  {np.mean(mae_scores):.4f} ± {np.std(mae_scores):.4f}")
print(f"R²:   {np.mean(r2_scores):.4f} ± {np.std(r2_scores):.4f}")

In [ ]:
import numpy as np
from sklearn.metrics import pairwise_distances
from sklearn.linear_model import LinearRegression

class LocalLinearRegression:
    def __init__(self, k=5):
        self.k = k

    def fit(self, X, y):
        self.X_train = np.asarray(X)
        self.y_train = np.asarray(y).ravel()
        return self

    def predict(self, X_test):
        X_test = np.asarray(X_test)
        n_test = X_test.shape[0]
        preds = np.zeros(n_test, dtype=float)

        dists = pairwise_distances(X_test, self.X_train)

        for i in range(n_test):
            idx = np.argsort(dists[i])[:self.k]
            X_neighbors = self.X_train[idx]
            y_neighbors = self.y_train[idx]

            lr = LinearRegression()
            lr.fit(X_neighbors, y_neighbors)
            preds[i] = lr.predict(X_test[i].reshape(1, -1))[0]

        return preds


In [ ]:
for k in (list(range(1, 11, 1)) + list(range(100, 1100, 100))):
    llr_model = LocalLinearRegression(k=k)
    
    llr_model.fit(
        X_train, y_train
    )
    
    y_pred_llr = llr_model.predict(X_test)
    
    rmse_llr = np.sqrt(mean_squared_error(y_test, y_pred_llr))
    mae_llr = mean_absolute_error(y_test, y_pred_llr)
    r2_llr = r2_score(y_test, y_pred_llr)
    
    print(f"k={k}")
    print(f"RMSE: {rmse_llr:.4f}")
    print(f"MAE:  {mae_llr:.4f}")
    print(f"R²:   {r2_llr:.4f}")
    print()
